# DICOM Preprocessing - Stub / Guarded Demonstration

Some clinical data (e.g. `data/raw/hospital_test/`) arrives as **DICOM series**
rather than NIfTI volumes. This notebook scaffolds that pipeline:

1. Detect whether any DICOM data is present under `data/raw/`
2. If present, load a series via SimpleITK and extract axial slices
3. Reuse the existing `resize_slice` / `min_max_normalize` transforms
4. Feed the result into the same loader chain (3-channel expansion)

**Current state:** no DICOM files are in the repo, so this notebook runs in
"stub mode" (it checks availability and stops gracefully). Re-run it once the
DICOM dataset is dropped in.

In [1]:
import sys
from pathlib import Path

import numpy as np
import yaml


def _find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for _ in range(10):
        if (current / "configs" / "config.yaml").exists():
            return current
        current = current.parent
    raise SystemExit(
        "Could not locate configs/config.yaml. Run the notebook from "
        "anywhere inside the repository."
    )

REPO_ROOT = _find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

with open(REPO_ROOT / "configs" / "config.yaml", "r") as fh:
    config = yaml.safe_load(fh)

RAW_DIR = REPO_ROOT / config["data"]["raw_dir"]
target_size = tuple(config["preprocessing"]["image_size"])

print("Config loaded.")

Config loaded.


In [2]:
dcm_files = sorted(RAW_DIR.rglob("*.dcm"))
dicomdirs = sorted(RAW_DIR.rglob("DICOMDIR"))

print(f"DICOM files found : {len(dcm_files)}")
print(f"DICOMDIRs found   : {len(dicomdirs)}")

if dcm_files or dicomdirs:
    print("MODE: DICOM data present - full demo will run.")
else:
    print("MODE: stub - no DICOM data present yet. The cells below document the intended pipeline.")

DICOM files found : 0
DICOMDIRs found   : 0
MODE: stub - no DICOM data present yet. The cells below document the intended pipeline.


## Intended DICOM pipeline

A DICOM **series** = a folder of `.dcm` files, one per axial slice. The planned
workflow mirrors the NIfTI one:

| Step | DICOM equivalent |
|---|---|
| Load the volume | `ImageSeriesReader` + `GetGDCMSeriesFileNames` -> `sitk.Image` |
| To numpy | `sitk.GetArrayFromImage(volume)` -> `(n_slices, H, W)` |
| Per-slice | each frame is a 2D grayscale image |
| Resize | `resize_slice(slice, (224, 224))` (unchanged) |
| Normalize | `min_max_normalize(slice)` (unchanged) |
| Loader | `to_three_channels` + `build_dataset` (unchanged) |

New code will land in `src/preprocessing/dicom.py` (currently an empty
placeholder with no functions).

In [3]:
if dcm_files or dicomdirs:
    import SimpleITK as sitk

    series_dir = dcm_files[0].parent if dcm_files else dicomdirs[0].parent

    series_ids = sitk.ImageSeriesReader.GetGDCMSeriesIDs(str(series_dir))
    print(f"Series IDs: {series_ids}")

    if series_ids:
        reader = sitk.ImageSeriesReader()
        reader.SetFileNames(
            sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(series_dir), series_ids[0])
        )
        image = reader.Execute()                      # sitk.Image
        volume = sitk.GetArrayFromImage(image)        # (n_slices, H, W)
        print(f"Volume shape: {volume.shape}, dtype: {volume.dtype}")
        print("Full demo would continue with resize_slice / min_max_normalize.")
    else:
        print("DICOM files found but no series recognized - nothing to demo.")
else:
    print("STUB MODE: nothing to load. Drop a DICOM series into data/raw/ and re-run.")

STUB MODE: nothing to load. Drop a DICOM series into data/raw/ and re-run.


## What happens when real DICOM arrives

1. Place the series under `data/raw/hospital_test/` (or any folder below `data/raw/`)
2. Re-run this notebook - `SimpleITK` will detect the series and exercise the demo
3. Decide whether to keep DICOM handling as a one-off script (`src/preprocess.py`)
   or a parallel `process_dicom` pipeline emitting the same `(224, 224)` float32
   arrays that `to_three_channels` / `build_augmentation_layer` already consume

Until then, `src/preprocessing/dicom.py` stays as an empty placeholder and
The current development experiment uses T2 NIfTI volumes from UT-EndoMRI and the MOGaMBO proxy-comparator cohort.
reason neither class has negatives yet.